# 03 — Anomaly Detection in Sensor Events

This notebook builds an unsupervised anomaly detection workflow for operational sensor data.

The hidden anomaly labels are used only after scoring, as an offline evaluation tool.


In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.covariance import EllipticEnvelope
from sklearn.decomposition import PCA
from sklearn.ensemble import IsolationForest
from sklearn.metrics import precision_score
from sklearn.neighbors import LocalOutlierFactor
from sklearn.preprocessing import StandardScaler
from sklearn.svm import OneClassSVM

sys.path.append(str(Path.cwd() / "src"))

from unsup_lab.data import make_sensor_anomaly_data


In [ ]:
dataset = make_sensor_anomaly_data(n_points=3_000, contamination=0.04, random_state=42)
raw = dataset.features
labels = dataset.hidden_labels

raw.head()


In [ ]:
feature_columns = [
    "temperature",
    "motion_count",
    "power_usage",
    "signal_strength",
    "missing_ratio",
]

x = StandardScaler().fit_transform(raw[feature_columns])
y_true = (labels == "anomaly").astype(int).to_numpy()


## Score multiple anomaly detectors

In [ ]:
scores = pd.DataFrame(index=raw.index)

isolation = IsolationForest(contamination=0.04, random_state=42)
isolation.fit(x)
scores["isolation_forest"] = -isolation.score_samples(x)

lof = LocalOutlierFactor(n_neighbors=35, contamination=0.04)
lof_labels = lof.fit_predict(x)
scores["local_outlier_factor"] = -lof.negative_outlier_factor_

robust_cov = EllipticEnvelope(contamination=0.04, random_state=42)
robust_cov.fit(x)
scores["robust_covariance"] = -robust_cov.score_samples(x)

svm = OneClassSVM(nu=0.04, gamma="scale")
svm.fit(x)
scores["one_class_svm"] = -svm.score_samples(x)

pca = PCA(n_components=2, random_state=42)
x_low = pca.fit_transform(x)
x_reconstructed = pca.inverse_transform(x_low)
scores["pca_reconstruction"] = np.mean((x - x_reconstructed) ** 2, axis=1)

scores.head()


## Precision at top-k

In [ ]:
def precision_at_k(score_values, y_true, k):
    top_indices = np.argsort(score_values)[-k:]
    return y_true[top_indices].mean()

k = int(y_true.sum())
results = {
    column: precision_at_k(scores[column].to_numpy(), y_true, k)
    for column in scores.columns
}

pd.Series(results, name=f"precision_at_{k}").sort_values(ascending=False)


## Review top anomalies

In [ ]:
best_score = scores.mean(axis=1)
review = raw.copy()
review["anomaly_score"] = best_score
review["hidden_label_for_offline_eval"] = labels

review.sort_values("anomaly_score", ascending=False).head(10)


## Limitations

Anomaly scores are not probabilities. Thresholds should be calibrated with analyst feedback, operational cost, and tolerance for false positives.
